In [ ]:
import os

import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import MLFlowLogger
from torch.utils.data import DataLoader
from torchvision.transforms import v2

from dataset import SegmentationDataset
from model import LeafModel

# Enable Tensor Cores on A100 for ~2x speed boost
torch.set_float32_matmul_precision("medium")

%matplotlib inline
%load_ext autoreload
%autoreload 2

Create datasets and dataloaders

In [ ]:
# We have to resize the images and masks to a fixed size for batching
dataset_path = "roboflow_tomato/"

transforms = v2.Compose(
    [
        v2.Resize((256, 256), interpolation=v2.InterpolationMode.NEAREST),
    ]
)

train_dataset = SegmentationDataset(
    root_dir=f"{dataset_path}/train/", transform=transforms, target_transform=transforms
)
val_dataset = SegmentationDataset(root_dir=f"{dataset_path}/valid/", transform=transforms, target_transform=transforms)
test_dataset = SegmentationDataset(root_dir=f"{dataset_path}/test/", transform=transforms, target_transform=transforms)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

n_cpu = os.cpu_count() or 8
# Increased batch size from 32 to 64 for fewer iterations and better GPU utilization
train_dataloader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=n_cpu - 2, pin_memory=True, persistent_workers=True
)
valid_dataloader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=n_cpu - 2, pin_memory=True, persistent_workers=True
)
test_dataloader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, num_workers=n_cpu - 2, pin_memory=True, persistent_workers=True
)

Visualize a few samples from the dataset to ensure that images and masks are loaded correctly.

In [ ]:
num_samples = 2

figure, axes = plt.subplots(num_samples, 2, figsize=(10, 4 * num_samples))
figure.suptitle("Sample Images and Masks from Training Set", fontsize=16, y=0.995)

for i in range(num_samples):
    img_id = torch.randint(0, len(train_dataset), (1,)).item()
    image, mask = train_dataset[img_id]

    # Display image
    axes[i, 0].imshow(image.permute(1, 2, 0))
    axes[i, 0].set_title(f"Image {i + 1}")
    axes[i, 0].axis("off")

    # Display mask
    axes[i, 1].imshow(mask.squeeze(), cmap="gray")
    axes[i, 1].set_title(f"Mask {i + 1}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

Enable mlflow logging

In [ ]:
# Create temporary directory for local artifacts
# temp_dir = tempfile.mkdtemp()

os.environ["MLFLOW_TRACKING_TOKEN"] = ""

mlflow_logger = MLFlowLogger(
    experiment_name="leaf_segmentation_tomato",
    tracking_uri="https://gitlab.com/api/v4/projects/72607084/ml/mlflow/",
    log_model=True,
    # save_dir=temp_dir,  # Temporary local storage
)

Configure model

In [ ]:
EPOCHS = 300
T_MAX = EPOCHS * len(train_dataloader)
OUT_CLASSES = 1
model = LeafModel("FPN", "resnet34", in_channels=3, out_classes=1, t_max=T_MAX)

# Optional: Compile model for faster training (PyTorch 2.0+)
# Uncomment if you have PyTorch 2.0 or later
# model = torch.compile(model)

Set hyperparameters and start training loop

In [ ]:
# Early stopping callback - monitors validation loss
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,  # Increased patience for 300 epochs
    mode="min",
    verbose=True,
)

# Model checkpoint callback - only saves best model to MLflow
checkpoint_callback = ModelCheckpoint(
    # dirpath=temp_dir,  # Temporary local storage
    filename="best-model",
    monitor="val_loss",
    mode="min",
    save_top_k=1,  # Only keep the best model
    save_last=False,  # Don't save the last model
    every_n_epochs=1,  # Check only at end of each epoch (not every batch)
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    callbacks=[early_stop_callback, checkpoint_callback],
    enable_checkpointing=True,
)

trainer.fit(
    model,
    train_dataloaders=train_dataloader,
    val_dataloaders=valid_dataloader,
)

# Clean up temporary directory after training
# shutil.rmtree(temp_dir, ignore_errors=True)

Validation Metrics

In [ ]:
valid_metrics = trainer.validate(model, dataloaders=valid_dataloader, verbose=False)
print(valid_metrics)

Visualize predictions

In [ ]:
import random

num_samples = 10
# Get all test samples
all_samples = list(range(len(test_dataset)))
random_indices = random.sample(all_samples, min(num_samples, len(all_samples)))

for idx in random_indices:
    image, gt_mask = test_dataset[idx]

    with torch.inference_mode():
        model.eval()
        logits = model(image.unsqueeze(0))

    pr_mask = logits.sigmoid().squeeze()

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(image.permute(1, 2, 0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask.squeeze(), cmap="gray")
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pr_mask.detach().numpy(), cmap="gray")
    plt.title("Prediction")
    plt.axis("off")
    plt.show()